# 02_data_understanding

Objetivo desta etapa: entender a estrutura dos dados da CVM (DFP e ITR) antes de qualquer tratamento.

1. Dimensão geral da base
2. Definição do identificador único de companhia
3. Inventário estrutural dos arquivos (linhas, colunas, tipos)
4. Validação de consistência entre arquivos

In [1]:
%pip install duckdb
import hashlib
from pathlib import Path

import duckdb
import pandas as pd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuração de caminhos

Focando inicialmente nos arquivos **DFP**. Para repetir a análise no ITR,
basta trocar `FOCO` para `"itr"`.

In [2]:
FOCOS = ["dfp", "itr"] 

pasta_raiz = Path.cwd().parent / "data/interim"

pastas = {
    "dfp": pasta_raiz / "02_dfp_concatenated",
    "itr": pasta_raiz / "02_itr_concatenated",
}

caminho_atual = Path.cwd()
pasta_anterior = caminho_atual.parent
arquivo_exemplo = pasta_anterior / "data/interim/02_itr_concatenated/itr_cia_aberta_DRE_con_2011-2025.parquet"

## Dimensão geral da base (arquivo de exemplo)

Antes de qualquer análise, verifica-se a quantidade total de registros
para compreender a dimensão dos dados.

In [3]:
duckdb.sql(f"""
    SELECT COUNT(*) AS total_registros
    FROM '{arquivo_exemplo}'
""").df()

,total_registros
0,1991007


## Identificador único de companhia

Para construir os bancos de dados, é preciso identificar uma empresa
de forma exclusiva. Comparamos três candidatos: CNPJ, razão social e código CVM.

In [4]:
duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT CNPJ_CIA) AS cnpjs,
        COUNT(DISTINCT DENOM_CIA) AS razoes_sociais,
        COUNT(DISTINCT CD_CVM)   AS codigos_cvm
    FROM '{arquivo_exemplo}'
""").df()

,cnpjs,razoes_sociais,codigos_cvm
0,721,756,725


As quantidades diferem porque:

- **Razões sociais > CNPJs/CD_CVM**: companhias mudam de nome ao longo do tempo
  (fusões, aquisições, reestruturações societárias).
- **CD_CVM vs CNPJ**: precisamos checar se há relação 1:1 entre eles.

In [5]:
# Um mesmo CD_CVM está associado a mais de um CNPJ?
duckdb.sql(f"""
    SELECT CD_CVM, COUNT(DISTINCT CNPJ_CIA) AS qtd_cnpjs
    FROM '{arquivo_exemplo}'
    GROUP BY CD_CVM
    HAVING COUNT(DISTINCT CNPJ_CIA) > 1
    ORDER BY qtd_cnpjs DESC
""").df()

,CD_CVM,qtd_cnpjs


In [6]:
# Um mesmo CNPJ está associado a mais de um CD_CVM?
duckdb.sql(f"""
    SELECT CNPJ_CIA, COUNT(DISTINCT CD_CVM) AS qtd_codigos
    FROM '{arquivo_exemplo}'
    GROUP BY CNPJ_CIA
    HAVING COUNT(DISTINCT CD_CVM) > 1
    ORDER BY qtd_codigos DESC
""").df()

,CNPJ_CIA,qtd_codigos
0,62.258.884/0001-36,2
1,08.926.302/0001-05,2
2,09.149.503/0001-06,2
3,59.717.553/0001-02,2


**Conclusão:** cada CD_CVM está associado a apenas um CNPJ, mas alguns CNPJs
possuem mais de um CD_CVM (reorganizações societárias). Como todo o ecossistema
regulatório da CVM gira em torno do código CVM, **ele será usado como
identificador exclusivo da companhia**.

Para a tabela mestre de empresas, será usada a razão social mais recente
disponível; as demonstrações financeiras preservam a razão social vigente
em cada data de referência.

> Empresas sem negociação relevante na bolsa serão filtradas depois via
> séries históricas da B3 (COTAHIST), não pelo cadastro da CVM.

## Inventário dos arquivos

Nesta etapa é construído um inventário com metadados de cada arquivo
(origem, tipo de demonstrativo, consolidação, linhas, colunas, tipos)
para servir de referência nas etapas de auditoria.

In [7]:
def parse_nome_arquivo(nome: str) -> dict:
    """Extrai metadados a partir do padrão de nome dos arquivos da CVM."""
    origem = "dfp" if nome.startswith("dfp") else "itr"

    if "_con_" in nome:
        consolidacao = "con"
    elif "_ind_" in nome:
        consolidacao = "ind"
    else:
        consolidacao = "NA"

    partes_conhecidas = [
        "BPA", "BPP", "DFC_MD", "DFC_MI", "DMPL",
        "DRA", "DRE", "DVA", "composicao_capital", "parecer",
    ]
    demonstrativo = next((p for p in partes_conhecidas if p in nome), "principal")

    return {
        "origem": origem,
        "demonstrativo": demonstrativo,
        "consolidacao": consolidacao,
    }

In [8]:
def construir_inventario(arquivos: list[Path]) -> tuple[pd.DataFrame, dict]:
    schemas = {}
    todas_colunas = set()

    for arq in arquivos:
        schema = duckdb.sql(f"DESCRIBE SELECT * FROM '{arq}'").df()
        schemas[arq.name] = schema
        todas_colunas.update(schema["column_name"])

    # estrutura mais comum (moda) entre os arquivos
    estruturas = [tuple(s["column_name"]) for s in schemas.values()]
    estrutura_padrao = max(set(estruturas), key=estruturas.count)

    linhas_inventario = []

    for arq in arquivos:
        schema = schemas[arq.name]
        colunas = list(schema["column_name"])
        metadados = parse_nome_arquivo(arq.name)

        linhas_inventario.append({
            "arquivo": arq.name,
            **metadados,
            "tamanho_mb": round(arq.stat().st_size / 1e6, 2),
            "linhas": duckdb.sql(f"SELECT COUNT(*) FROM '{arq}'").fetchone()[0],
            "n_colunas": len(colunas),
            "estrutura_padrao": tuple(colunas) == estrutura_padrao,
            "colunas": "; ".join(colunas),
            "tipos": "; ".join(schema["column_name"] + " (" + schema["column_type"] + ")"),
            "colunas_ausentes": "; ".join(sorted(todas_colunas - set(colunas))),
            "qtd_colunas_ausentes": len(todas_colunas - set(colunas)),
            "hash_estrutura": hashlib.md5("|".join(colunas).encode()).hexdigest(),
        })

    return pd.DataFrame(linhas_inventario), schemas

In [9]:
inventarios = []
schemas_todos = {}

for foco in FOCOS:
    pasta_foco = pastas[foco]
    arquivos = sorted(pasta_foco.glob("*.parquet"))

    inv, schemas = construir_inventario(arquivos)
    inventarios.append(inv)
    schemas_todos[foco] = schemas

    print(f"{foco}: {len(arquivos)} arquivos processados")

inventario = pd.concat(inventarios, ignore_index=True)
inventario

dfp: 19 arquivos processados
itr: 19 arquivos processados


,arquivo,origem,demonstrativo,consolidacao,tamanho_mb,linhas,n_colunas,estrutura_padrao,colunas,tipos,colunas_ausentes,qtd_colunas_ausentes,hash_estrutura
0,dfp_cia_aberta_2010-2025.parquet,dfp,principal,NA,0.37,13994,13,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CD_CONTA; COLUNA_DF; DS_CONTA; DT_FIM_EXERC; D...,21,0f9770bdb26a745d24cd4875263c1921
1,dfp_cia_aberta_BPA_con_2010-2025.parquet,dfp,BPA,con,3.87,821913,18,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_INI_EXERC; DT_RECEB; ...,16,dace2a105d18960b1ef3c13c59786a04
2,dfp_cia_aberta_BPA_ind_2010-2025.parquet,dfp,BPA,ind,5.40,1408072,18,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_INI_EXERC; DT_RECEB; ...,16,dace2a105d18960b1ef3c13c59786a04
3,dfp_cia_aberta_BPP_con_2010-2025.parquet,dfp,BPP,con,5.61,1398939,18,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_INI_EXERC; DT_RECEB; ...,16,dace2a105d18960b1ef3c13c59786a04
4,dfp_cia_aberta_BPP_ind_2010-2025.parquet,dfp,BPP,ind,7.82,2331144,18,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_INI_EXERC; DT_RECEB; ...,16,dace2a105d18960b1ef3c13c59786a04
5,dfp_cia_aberta_composicao_capital_2010-2025.pa...,dfp,composicao_capital,NA,0.11,4343,14,False,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; QT_ACAO...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; CD_CONTA; CD_CVM; COLUNA_DF; DS_CON...,20,0398b13d15be5e271ad2a1a94b728c66
6,dfp_cia_aberta_DFC_MD_con_2010-2025.parquet,dfp,DFC_MD,con,0.08,8570,19,True,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_RECEB; ID_DOC; LINK_D...,15,7052ca5656f1c41c689a1f40cf6fa5ba
7,dfp_cia_aberta_DFC_MD_ind_2010-2025.parquet,dfp,DFC_MD,ind,0.10,13114,19,True,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_RECEB; ID_DOC; LINK_D...,15,7052ca5656f1c41c689a1f40cf6fa5ba
8,dfp_cia_aberta_DFC_MI_con_2010-2025.parquet,dfp,DFC_MI,con,7.58,614288,19,True,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_RECEB; ID_DOC; LINK_D...,15,7052ca5656f1c41c689a1f40cf6fa5ba
9,dfp_cia_aberta_DFC_MI_ind_2010-2025.parquet,dfp,DFC_MI,ind,10.08,855335,19,True,CNPJ_CIA; DT_REFER; VERSAO; DENOM_CIA; CD_CVM;...,CNPJ_CIA (VARCHAR); DT_REFER (VARCHAR); VERSAO...,CATEG_DOC; COLUNA_DF; DT_RECEB; ID_DOC; LINK_D...,15,7052ca5656f1c41c689a1f40cf6fa5ba


## Comparação estrutural entre arquivos

Arquivos do mesmo `demonstrativo` devem ter exatamente o mesmo conjunto
de colunas. Aqui verificamos divergências agrupando por hash de estrutura.

In [10]:
(
    inventario
    .groupby(["demonstrativo", "hash_estrutura"])
    .agg(arquivos=("arquivo", list), n_colunas=("n_colunas", "first"))
    .reset_index()
)

,demonstrativo,hash_estrutura,arquivos,n_colunas
0,BPA,dace2a105d18960b1ef3c13c59786a04,"[dfp_cia_aberta_BPA_con_2010-2025.parquet, dfp...",18
1,BPP,dace2a105d18960b1ef3c13c59786a04,"[dfp_cia_aberta_BPP_con_2010-2025.parquet, dfp...",18
2,DFC_MD,7052ca5656f1c41c689a1f40cf6fa5ba,"[dfp_cia_aberta_DFC_MD_con_2010-2025.parquet, ...",19
3,DFC_MI,7052ca5656f1c41c689a1f40cf6fa5ba,"[dfp_cia_aberta_DFC_MI_con_2010-2025.parquet, ...",19
4,DMPL,27908ea11c405a4eb22daeb9ce82b6c7,"[dfp_cia_aberta_DMPL_con_2010-2025.parquet, df...",20
5,DRA,7052ca5656f1c41c689a1f40cf6fa5ba,"[dfp_cia_aberta_DRA_con_2010-2025.parquet, dfp...",19
6,DRE,7052ca5656f1c41c689a1f40cf6fa5ba,"[dfp_cia_aberta_DRE_con_2010-2025.parquet, dfp...",19
7,DVA,7052ca5656f1c41c689a1f40cf6fa5ba,"[dfp_cia_aberta_DVA_con_2010-2025.parquet, dfp...",19
8,composicao_capital,0398b13d15be5e271ad2a1a94b728c66,[dfp_cia_aberta_composicao_capital_2010-2025.p...,14
9,parecer,53a551d0e13988bc4c5b3880cdfd61e7,[dfp_cia_aberta_parecer_2010-2025.parquet],12


In [11]:
# Arquivos fora da estrutura mais comum global (ajuda a achar outliers reais)
inventario[~inventario["estrutura_padrao"]][["arquivo", "demonstrativo", "n_colunas"]]

,arquivo,demonstrativo,n_colunas
0,dfp_cia_aberta_2010-2025.parquet,principal,13
1,dfp_cia_aberta_BPA_con_2010-2025.parquet,BPA,18
2,dfp_cia_aberta_BPA_ind_2010-2025.parquet,BPA,18
3,dfp_cia_aberta_BPP_con_2010-2025.parquet,BPP,18
4,dfp_cia_aberta_BPP_ind_2010-2025.parquet,BPP,18
5,dfp_cia_aberta_composicao_capital_2010-2025.pa...,composicao_capital,14
10,dfp_cia_aberta_DMPL_con_2010-2025.parquet,DMPL,20
11,dfp_cia_aberta_DMPL_ind_2010-2025.parquet,DMPL,20
18,dfp_cia_aberta_parecer_2010-2025.parquet,parecer,12
19,itr_cia_aberta_2011-2025.parquet,principal,13


## Consistência de tipos por coluna

Verifica se alguma coluna aparece com tipos diferentes entre arquivos
(ex.: `VL_CONTA` como DOUBLE em um arquivo e VARCHAR em outro).

In [12]:
tipos_por_arquivo = pd.concat(
    [
        schema[["column_name", "column_type"]].assign(arquivo=nome)
        for schemas in schemas_todos.values()
        for nome, schema in schemas.items()
    ],
    ignore_index=True,
)

colunas_com_tipos_divergentes = (
    tipos_por_arquivo
    .groupby("column_name")["column_type"]
    .nunique()
    .loc[lambda s: s > 1]
)

colunas_com_tipos_divergentes

Series([], Name: column_type, dtype: int64)

In [13]:
for coluna in colunas_com_tipos_divergentes.index:
    print("=" * 80)
    print(coluna)
    display(
        tipos_por_arquivo
        .loc[tipos_por_arquivo["column_name"] == coluna, ["arquivo", "column_type"]]
        .sort_values("arquivo")
    )

## Salvando o inventário consolidado

Este arquivo servirá de referência na etapa de auditoria, para validar
que o layout esperado permaneceu inalterado.

In [14]:
caminho_saida = pasta_raiz / "03_dfp_itr_inventory.csv"

inventario.to_csv(caminho_saida, index=False, sep=";")

caminho_saida

WindowsPath('c:/Users/paulo/Desktop/Python/brazilian_financial_database/data/interim/03_dfp_itr_inventory.csv')